# Analysis of the isolated MW-like galaxy (`examples/mw_gas`)

This notebook reads Gadget4 snapshots with **pynbody** and produces:

1. face-on and edge-on density maps of stars and gas;
2. the rotation curve, by component;
3. surface-density profiles at several times;
4. gas and stellar mass versus time;
5. a disk-stability summary: bar strength and thickness versus time.
6. an equilibrium check of the initial conditions: radial "breathing" of the stellar disk.

**Run it from the `notebooks/` directory.** pynbody reads `./config.ini` at import time. That file
maps Gadget PartType2/3 (stellar disk, bulge) to the `star` family; pynbody's default would put
them in `dm`.

See `tutorials/04_analysis_pynbody.md` for explanations.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pynbody

import mwtools   # helpers in this directory: load(), snapshots(), time_myr()

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams["figure.dpi"] = 90

RUN = os.environ.get("MW_RUN", "mw_gas")      # or "mw_gas_test"
files = mwtools.snapshots(RUN)
print(f"{RUN}: {len(files)} snapshots in {mwtools.run_dir(RUN)}")

## 1. Loading a snapshot

`mwtools.load()` wraps `pynbody.load()` and does four things:

* **Units.** It converts to physical units (kpc, Msun, km/s).
* **Time.** It sets `s.properties['time']` from the HDF5 header. For a non-cosmological run,
  pynbody would otherwise derive an "age of the universe" (~9.8 Gyr) from the empty cosmology.
* **Components.** It adds `s['comp']`: 0 gas, 1 DM halo, 2 stellar disk, 3 bulge. InkWell numbers
  particle IDs consecutively by type, and Gadget4 keeps the IDs.
* **Orientation.** It centres on the stars and rotates the snapshot face-on
  (`pynbody.analysis.faceon`), so the disk lies in the x–y plane.

In [ ]:
s = mwtools.load(files[-1])
print(s, f"t = {mwtools.time_myr(s):.0f} Myr")
print("families:", {f.name: len(s[f]) for f in s.families()})
print("components (gas, DM, disk, bulge):", np.bincount(s["comp"]))
print("arrays on disk:", s.loadable_keys())

disk, bulge = s.s[s.s["comp"] == 2], s.s[s.s["comp"] == 3]
print(f"M_gas = {s.g['mass'].sum():.3e}, M_disk = {disk['mass'].sum():.3e}, "
      f"M_bulge = {bulge['mass'].sum():.3e}, M_DM = {s.dm['mass'].sum():.3e} Msun")

## 2. Face-on and edge-on density maps

`pynbody.plot.sph.image` projects any family with an SPH kernel. For stars it first computes
smoothing lengths with a KD-tree, which takes a minute for millions of particles. Projecting `rho`
with units of Msun kpc⁻² gives the surface density.

`pynbody.analysis.sideon` rotates the disk edge-on; the rotation is undone afterwards.

In [ ]:
def maps(s, width=40):
    fig, ax = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)
    for row, view in enumerate(["face-on", "edge-on"]):
        if view == "edge-on":
            tr = pynbody.analysis.sideon(s, disk_size="5 kpc")
        for col, (fam, lab, vr) in enumerate([(s.s, "stars", (1e5, 1e10)), (s.g, "gas", (1e4, 1e8))]):
            pynbody.plot.sph.image(fam, qty="rho", units="Msol kpc^-2", width=width, cmap="magma",
                                   vmin=vr[0], vmax=vr[1], axes=ax[row, col], show_cbar=True)
            ax[row, col].set_title(f"{lab}, {view}, t = {mwtools.time_myr(s):.0f} Myr")
        if view == "edge-on":
            tr.revert()
    return fig

maps(mwtools.load(files[0]));
maps(s);

## 3. Rotation curve

There are two complementary measures.

* **Circular velocity** v_c(r) = sqrt(G M(<r) / r) per component, from the enclosed mass. This is
  exact for the spherical halo and bulge, and a good approximation for the disk beyond ~2 R_d.
  `pynbody.analysis.profile.Profile(..., type='log')` gives the enclosed mass `mass_enc`.
* **Mean rotation velocity** ⟨v_φ⟩ of stars and gas in the plane (`vphi` in a 2D profile).
  The stars lag v_c because of asymmetric drift (random motions); the cold gas should follow v_c.

In [ ]:
G = 4.30091e-6  # kpc (km/s)^2 / Msun

def rotation_curve(s, ax, rmax=30):
    kw = dict(rmin=0.1, rmax=rmax, nbins=60, type="log", ndim=3)
    comps = {"gas": s.g, "DM halo": s.dm, "stellar disk": s.s[s.s["comp"] == 2],
             "bulge": s.s[s.s["comp"] == 3]}
    vc2 = 0
    for lab, sub in comps.items():
        p = pynbody.analysis.profile.Profile(sub, **kw)
        v2 = G * p["mass_enc"].in_units("Msol") / p["rbins"].in_units("kpc")
        vc2 = vc2 + v2
        ax.plot(p["rbins"], np.sqrt(v2), lw=1, label=lab)
    ax.plot(p["rbins"], np.sqrt(vc2), "k", lw=2, label=r"total $v_c$")
    for fam, ls, lab in [(s.s[s.s["comp"] == 2], "--", "stars"), (s.g, ":", "gas")]:
        sel = fam[pynbody.filt.BandPass("z", "-1 kpc", "1 kpc")]
        p2 = pynbody.analysis.profile.Profile(sel, rmin=0.5, rmax=rmax, nbins=40, ndim=2)
        ax.plot(p2["rbins"], p2["vphi"].in_units("km s^-1"), "k" + ls, label=rf"{lab} $\langle v_\phi\rangle$")
    ax.set(xlabel="R [kpc]", ylabel="v [km/s]", xlim=(0, rmax), ylim=(0, 300),
           title=f"t = {mwtools.time_myr(s):.0f} Myr")
    ax.legend(fontsize=8)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True, sharey=True)
rotation_curve(mwtools.load(files[0]), ax[0])
rotation_curve(s, ax[1])

## 4. Surface-density profiles

A 2D `Profile` (`ndim=2`) bins in cylindrical R. Its `density` is then a surface density. An
exponential disk is a straight line in a log-linear plot; its slope gives −1/R_d.

In [ ]:
idx = sorted(set(np.linspace(0, len(files) - 1, min(5, len(files))).astype(int)))
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for i in idx:
    si = mwtools.load(files[i])
    t = mwtools.time_myr(si)
    for a, sub in zip(ax, [si.s[si.s["comp"] == 2], si.g, si.s[si.s["comp"] == 3]]):
        p = pynbody.analysis.profile.Profile(sub, rmin=0.2, rmax=25, nbins=50, ndim=2)
        a.semilogy(p["rbins"], p["density"].in_units("Msol pc^-2"), label=f"{t:.0f} Myr")
for a, lab in zip(ax, ["stellar disk", "gas", "bulge"]):
    a.set(xlabel="R [kpc]", ylabel=r"$\Sigma$ [M$_\odot$ pc$^{-2}$]", title=lab)
    a.legend(fontsize=8)

# exponential scale length of the stellar disk, fitted between 3 and 12 kpc, first vs last snapshot.
# (This simple fit gives ~3.3 kpc at t = 0 for the InkWell R_d = 3.0 disk: compare the two times, not the absolute value.)
def scale_length(snap):
    p = pynbody.analysis.profile.Profile(snap.s[snap.s["comp"] == 2], rmin=3, rmax=12, nbins=20, ndim=2)
    return -1 / np.polyfit(p["rbins"], np.log(p["density"]), 1)[0]
s0 = mwtools.load(files[0])
print(f"stellar disk R_d (fit 3-12 kpc): {scale_length(s0):.2f} kpc at t = 0 -> {scale_length(s):.2f} kpc at t = {mwtools.time_myr(s):.0f} Myr")

## 5. Gas and stellar mass versus time

Gadget4 writes the total mass of each particle type to `output/energy.txt` every
`TimeBetStatistics` (5 Myr). The last `NTYPES` columns are the masses. This gives a much finer
time series than the snapshots, which we also sum as a cross-check.

This run has no cooling or star formation (`ISOTHERM_EQS`), so both curves must stay **flat**:
this is a mass-conservation check. With `COOLING` + `STARFORMATION`, the gas mass would decrease
and the stellar mass (types 2–4) would increase.

In [ ]:
e = np.loadtxt(os.path.join(mwtools.run_dir(RUN), "energy.txt"))
ntypes = (e.shape[1] - 4) // 4                  # columns: t, Eth, Epot, Ekin, 3 x NTYPES, NTYPES masses
t_gyr = e[:, 0] * mwtools.TIME_UNIT_GYR
mtype = e[:, -ntypes:] * 1e10                   # Msun
m_gas, m_star = mtype[:, 0], mtype[:, 2:].sum(1)  # stars: disk + bulge (+ new stars, type 4)

snap_t, snap_g, snap_s = [], [], []
for f in files:
    si = mwtools.load(f, center=False)
    snap_t.append(mwtools.time_myr(si) / 1e3)
    snap_g.append(si.g["mass"].sum()); snap_s.append(si.s["mass"].sum())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_gyr, m_star, label="stars (energy.txt)")
ax.plot(t_gyr, m_gas, label="gas (energy.txt)")
ax.plot(snap_t, snap_s, "o", color="C0", label="stars (snapshots)")
ax.plot(snap_t, snap_g, "o", color="C1", label="gas (snapshots)")
ax.set(yscale="log", xlabel="t [Gyr]", ylabel=r"M [M$_\odot$]", ylim=(1e9, 1e11))
ax.legend()

E = e[:, 1] + e[:, 2] + e[:, 3]
print(f"gas mass change {m_gas[-1] / m_gas[0] - 1:+.1e}, stellar {m_star[-1] / m_star[0] - 1:+.1e}")
print(f"relative total-energy drift: {(E[-1] - E[0]) / abs(E[0]):+.2e}")

## 6. Disk stability: bar strength and thickness

The m = 2 Fourier amplitude A₂ = |Σ m e^{2iφ}| / Σ m of the stellar disk inside 6 kpc measures
bar strength: A₂ ≲ 0.1 means no bar, and A₂ > 0.2 means a clear bar.

The rms height at 7–9 kpc tracks vertical heating. Numerical heating by heavy DM particles would
show up here as a steady growth.

In [ ]:
rows = []
for f in files:
    si = mwtools.load(f)
    d = si.s[si.s["comp"] == 2]
    R, phi, m, z = np.hypot(d["x"], d["y"]), np.arctan2(d["y"], d["x"]), d["mass"], d["z"]
    inner = (R < 6) & (abs(z) < 1)
    ring = (R > 7) & (R < 9)
    Rg = np.hypot(si.g["x"], si.g["y"])
    rows.append((mwtools.time_myr(si),
                 abs(np.sum(m[inner] * np.exp(2j * phi[inner]))) / m[inner].sum(),
                 np.sqrt(np.mean(z[ring] ** 2)),
                 np.sqrt(np.mean(si.g["z"][(Rg > 7) & (Rg < 9)] ** 2))))
rows = np.array(rows)
fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax[0].plot(rows[:, 0], rows[:, 1], "o-"); ax[0].axhline(0.1, color="gray", ls=":")
ax[0].set(xlabel="t [Myr]", ylabel=r"$A_2$ (R < 6 kpc)", ylim=(0, 0.3), title="bar strength")
ax[1].plot(rows[:, 0], rows[:, 2], "o-", label="stellar disk")
ax[1].plot(rows[:, 0], rows[:, 3], "o-", label="gas")
ax[1].set(xlabel="t [Myr]", ylabel=r"$z_{\rm rms}$ at 7-9 kpc [kpc]", ylim=(0, None), title="thickness")
ax[1].legend()
print(np.round(rows, 3))

## 7. Are the initial conditions in equilibrium? Radial breathing of the disk

An equilibrium disk has no net radial motion: the mean radial velocity ⟨v_R⟩ in every ring stays
at zero (to within particle noise), and the mean rotation ⟨v_φ⟩ stays constant. If the stars start
with slightly too much or too little rotational support for the potential they actually feel, the
whole disk expands or contracts coherently and then oscillates at the epicyclic frequency. This
"breathing" shows up as ⟨v_R⟩ of several km/s with the same sign in neighbouring rings, and moves
mass between the inner and outer disk.

This was measured with InkWell's `check_disc_equilibrium.py` and two 10M-particle runs (see
`reports/inkwell_gadget4_retest2.md`):

* InkWell `4ab25f8` (static analytic disk in the potential): ⟨v_R⟩ up to +9 km/s at 23 Myr, a
  13% dip in the inner surface density and a ~125 Myr oscillation that settles by ~200 Myr.
* InkWell ≥ `3e5907e` (`agama.disc_iterations: 2`, the default): |⟨v_R⟩| ≤ 0.7 km/s at all
  radii and times.

The particle noise in ⟨v_R⟩ scales as σ_R/√N_ring, about 0.1 km/s at 10M particles and ~3 km/s
at 100k, so judge the test run by the absence of a *coherent* pattern rather than by zero.

In [ ]:
edges = [1, 2, 3, 4, 6, 8, 10]                     # ring edges [kpc]
rings = list(zip(edges[:-1], edges[1:]))
t_b, vr_b, vphi_b = [], [], []
for f in files:
    si = mwtools.load(f)                               # centred on the stars, face-on
    d = si.s[si.s["comp"] == 2]                        # stellar disk only
    R, vR, vphi = np.asarray(d["rxy"]), np.asarray(d["vrxy"]), np.asarray(d["vcxy"])
    t_b.append(mwtools.time_myr(si))
    vr_b.append([vR[(R > a) & (R < b)].mean() for a, b in rings])
    vphi_b.append(vphi[(R > 3) & (R < 6)].mean())
vr_b = np.array(vr_b)

fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for j, (a, b) in enumerate(rings):
    ax[0].plot(t_b, vr_b[:, j], "o-", ms=3, label=f"{a}-{b} kpc")
ax[0].axhline(0, color="gray", lw=0.8)
ax[0].set(xlabel="t [Myr]", ylabel=r"$\langle v_R \rangle$ [km/s]", title="mean radial velocity of the stellar disk")
ax[0].legend(fontsize=8, ncol=2)
ax[1].plot(t_b, np.array(vphi_b) / vphi_b[0], "o-", ms=3)
ax[1].set(xlabel="t [Myr]", ylabel="relative to t = 0", title=r"$\langle v_\phi \rangle$ at R = 3-6 kpc")
print(f"max |<v_R>| over all rings and snapshots: {np.abs(vr_b).max():.2f} km/s")
print(f"<v_phi>(3-6 kpc): {vphi_b[0]:.1f} km/s at t = 0, range {min(vphi_b):.1f}-{max(vphi_b):.1f} km/s")